In [ ]:

import torch
from transformers import pipeline

MODEL_NAME = "sshleifer/distilbart-cnn-12-6"

_summarizer = None


def get_summarizer():
    global _summarizer

    if _summarizer is None:
        device = 0 if torch.cuda.is_available() else -1

        _summarizer = pipeline(
            "summarization",
            model=MODEL_NAME,
            device=device
        )

    return _summarizer


def summarize_text(text, max_length=120, min_length=30):
    """Summarize long text in smaller segments."""
    text = text.strip()

    if not text:
        return ""

    summarizer = get_summarizer()

    # The tokenizer controls the actual model input length.
    tokenizer = summarizer.tokenizer
    token_ids = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    max_input_tokens = min(
        900,
        tokenizer.model_max_length
        if tokenizer.model_max_length < 100000
        else 900
    )

    summaries = []

    for start in range(0, len(token_ids), max_input_tokens):
        segment_ids = token_ids[start:start + max_input_tokens]

        segment = tokenizer.decode(
            segment_ids,
            skip_special_tokens=True
        ).strip()

        if not segment:
            continue

        # Short segments may not satisfy the requested minimum length.
        output = summarizer(
            segment,
            max_length=max_length,
            min_length=min(min_length, max(5, max_length - 5)),
            do_sample=False,
            truncation=True
        )

        summaries.append(output[0]["summary_text"])

    if len(summaries) == 1:
        return summaries[0]

    # Combine segment summaries and compress them into a final summary.
    combined = " ".join(summaries)

    if not combined:
        return ""

    return summarizer(
        combined,
        max_length=max_length,
        min_length=min(min_length, max(5, max_length - 5)),
        do_sample=False,
        truncation=True
    )[0]["summary_text"]
